In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import joblib

# 1. Laden des Datensatzes
df = pd.read_csv('Datasets/Titanic_train.csv')

# 2. Vorverarbeitung
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df['Age']      = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

le_gender   = LabelEncoder()
le_embarked = LabelEncoder()
df['Gender']   = le_gender.fit_transform(df['Gender'])    # female=0, male=1
df['Embarked'] = le_embarked.fit_transform(df['Embarked'])  # C=0, Q=1, S=2

print(df.head())
print("Shape:", df.shape)

### EDA Start

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# Klassenverteilung: Überlebt vs. Nicht überlebt
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Survived')
plt.xticks([0, 1], ['Nicht überlebt', 'Überlebt'])
plt.title('Verteilung: Überlebt / Nicht überlebt')
plt.xlabel('Klasse')
plt.ylabel('Anzahl')
plt.show()

In [ ]:
# Merkmalsverteilung nach Survived (Boxplot)
df_melted = df.melt(id_vars='Survived')
df_melted['value'] = pd.to_numeric(df_melted['value'], errors='coerce')

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_melted, x='variable', y='value', hue='Survived')
plt.xticks(rotation=45)
plt.title('Merkmalsverteilung nach Survived')
plt.legend(title='Survived', labels=['Nicht überlebt', 'Überlebt'])
plt.tight_layout()
plt.show()

In [ ]:
# Korrelationsmatrix
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Korrelationsmatrix der Titanic-Merkmale')
plt.tight_layout()
plt.show()

### EDA Ende

In [ ]:
# 3. Features (X) und Target (y) trennen
# Features: Pclass, Gender, Age, SibSp, Parch, Fare, Embarked  -> 7 Merkmale
X = df.drop('Survived', axis=1).values
y = df['Survived'].values

# 4. Aufteilen des Datensatzes (Train / Val / Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)
X_train, X_val,  y_train, y_val  = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

# 5. Umwandeln in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

print(X_train.shape, y_train.shape)
print(X_test.shape,  X_val.shape)

In [ ]:
# 6. Neuronales Netz definieren  (7 Eingabe-Features -> 16 Neuronen -> 2 Klassen)
net = nn.Sequential(
    nn.Linear(7,16),
    nn.ReLU(),
    nn.Linear(16,32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.01)
print(net)

In [ ]:
loss_history = []

# 7. Training
net.train()
for epoch in range(100):
    optimizer.zero_grad()
    outputs = net(X_train)
    loss    = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())

    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
    net.train()

# Loss-Kurve plotten
plt.figure(figsize=(8, 4))
plt.plot(loss_history, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Trainingsverlauf')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 8. Auswertung auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)

# 9. Modell und Encoder abspeichern
torch.save(net.state_dict(), 'Models/titanic_net_NextStep0.pth')
joblib.dump(le_gender,   'Models/le_gender_NextStep0.pkl')
joblib.dump(le_embarked, 'Models/le_embarked_NextStep0.pkl')
print("Modell gespeichert.")

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test.numpy(), predicted.numpy())

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Nicht überlebt', 'Überlebt'],
            yticklabels=['Nicht überlebt', 'Überlebt'])
plt.xlabel('Vorhergesagte Klasse')
plt.ylabel('Tatsächliche Klasse')
plt.title('Confusion Matrix: Titanic Survival')
plt.tight_layout()
plt.show()

In [ ]:
# Wiederladen des State-Dicts
import torch
import torch.nn as nn
import joblib

net = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Linear(16, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)
net.load_state_dict(torch.load('Models/titanic_net_NextStep0.pth', map_location=torch.device('cpu')))
le_gender   = joblib.load('Models/le_gender_NextStep0.pkl')
le_embarked = joblib.load('Models/le_embarked_NextStep0.pkl')

for k, v in net.named_parameters():
    print(k, v)
net.eval()

In [ ]:
# Vorhersage für einen einzelnen Passagier (manuelle Eingabe)
print("Geben Sie die Passagier-Merkmale ein:")
pclass   = int(input("Klasse (1 / 2 / 3): "))
gender   = input("Geschlecht (male / female): ").strip().lower()
age      = float(input("Alter: "))
sibsp    = int(input("Geschwister / Ehepartner an Bord (SibSp): "))
parch    = int(input("Eltern / Kinder an Bord (Parch): "))
fare     = float(input("Ticketpreis (Fare): "))
embarked = input("Einschiffungshafen (C / Q / S): ").strip().upper()

import numpy as np

gender_enc   = le_gender.transform([gender])[0]
embarked_enc = le_embarked.transform([embarked])[0]

inputs = torch.tensor(
    [[pclass, gender_enc, age, sibsp, parch, fare, embarked_enc]],
    dtype=torch.float32
)

with torch.no_grad():
    outputs      = net(inputs)
    _, predicted = torch.max(outputs, 1)

result = "Überlebt" if predicted.item() == 1 else "Nicht überlebt"
print(f"Vorhersage: {result}")

In [ ]:
# Klassenwahrscheinlichkeiten für die Vorhersage (inputs aus vorheriger Zelle)
import torch.nn.functional as F

with torch.no_grad():
    output       = net(inputs)
    _, max_index = torch.max(output, 1)
    print("Vorhergesagte Klasse:", "Überlebt" if max_index.item() == 1 else "Nicht überlebt")

probabilities = F.softmax(output, dim=1)
print(f"Wahrscheinlichkeit Nicht überlebt: {probabilities[0][0]:.4f}")
print(f"Wahrscheinlichkeit Überlebt:       {probabilities[0][1]:.4f}")